In [ ]:
# all import at once, cuz why not :)
import pandas as pd
import numpy as np # for random data generation
import matplotlib.pyplot as plt
import kagglehub
import os
import torch
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, LabelEncoder #import OneHotEncoder
from sklearn.model_selection import KFold ,StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from tqdm import tqdm
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from torch.utils.data import TensorDataset, DataLoader
from torchvision.datasets import MNIST
from torch.optim import Adam, AdamW
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from collections import Counter





from IPython.display import clear_output
%pip install kagglehub catboost lightgbm tqdm -q
%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

clear_output()
from catboost import CatBoostRegressor
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Dataset shape: {df_food.shape}")
df_food.head()

In [ ]:
df_food.info()

In [ ]:
df_food.describe()

In [ ]:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_food, "Delivery_Time")

In [ ]:
# Task 5: Write your code here:

In [ ]:
df_clean = df_food.drop(columns=['Order_ID'])

In [ ]:
# Weather, Traffic_Level, Time_of_Day are the objects
# Courier_Experience_yrs  , Delivery_Time are float32
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_food['Delivery_Time'].mean())

df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_food['Courier_Experience_yrs'].median())


print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
print('data before encoding:\n', df_clean.head()) #show before encoding


label_encoder = LabelEncoder()

for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = label_encoder.fit_transform(df_clean[col])

print('\nData after encoding:\n', df_clean.head()) #show after encoding

In [ ]:
df_clean.head()

df_clean.describe()

In [ ]:
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean

In [ ]:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for linear regression results for each fold
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []



In [ ]:
from sklearn.model_selection import KFold

# Use previously generated random data (example: regression data)

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)





In [ ]:
average_losses = kf.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# wallah i know how to but no time


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: